In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/DSN_Hackathon'   # <-- EDIT to your folder
OUTPUT_DIR = DATA_DIR

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb; HAS_LGB = True
except ImportError: HAS_LGB = False
try:
    import xgboost as xgb; HAS_XGB = True
except ImportError: HAS_XGB = False
try:
    from catboost import CatBoostRegressor; HAS_CAT = True
except ImportError: HAS_CAT = False

print("LightGBM:", HAS_LGB, "| XGBoost:", HAS_XGB, "| CatBoost:", HAS_CAT)

RNG = 42
N_FOLDS = 5
SEEDS = [42, 202, 777]   # bagging over multiple seeds cuts variance

LightGBM: True | XGBoost: True | CatBoost: False


In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

train['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = np.nan
full = pd.concat([train, test], ignore_index=True, sort=False)
print(train.shape, test.shape)

(6818, 14) (1705, 14)


In [ ]:
full['product_category'] = full['product_category'].str.strip().str.lower()
full['fat_content'] = full['fat_content'].str.strip().str.title()

cat_to_broad = {
    'health and hygiene': 'Non_Consumable', 'household': 'Non_Consumable', 'others': 'Non_Consumable',
    'soft drinks': 'Drinks', 'hard drinks': 'Drinks',
}
full['item_type_broad'] = full['product_category'].map(cat_to_broad).fillna('Food')
full.loc[full['item_type_broad'] == 'Non_Consumable', 'fat_content'] = 'Non_Edible'

In [ ]:
w_by_code = full.groupby('product_code')['product_weight_kg'].transform('mean')
full['product_weight_kg'] = full['product_weight_kg'].fillna(w_by_code)
w_by_cat = full.groupby('product_category')['product_weight_kg'].transform('mean')
full['product_weight_kg'] = full['product_weight_kg'].fillna(w_by_cat)
full['product_weight_kg'] = full['product_weight_kg'].fillna(full['product_weight_kg'].mean())

full['store_size'] = full['store_size'].fillna('Unknown')   # structural, whole-store missing

full['shelf_visibility_raw'] = full['shelf_visibility']
full.loc[full['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
vis_by_code = full.groupby('product_code')['shelf_visibility'].transform('mean')
full['shelf_visibility'] = full['shelf_visibility'].fillna(vis_by_code)
vis_by_cat = full.groupby('product_category')['shelf_visibility'].transform('mean')
full['shelf_visibility'] = full['shelf_visibility'].fillna(vis_by_cat)

In [ ]:
full['price_per_kg'] = full['product_price'] / full['product_weight_kg']
full['log_price'] = np.log1p(full['product_price'])
full['log_weight'] = np.log1p(full['product_weight_kg'])
full['visibility_mean_ratio'] = full['shelf_visibility'] / full.groupby('product_category')['shelf_visibility'].transform('mean')
full['visibility_rank_in_store'] = full.groupby('store_code')['shelf_visibility'].rank(pct=True)
full['price_rank_in_category'] = full.groupby('product_category')['product_price'].rank(pct=True)
full['price_vs_category_median'] = full['product_price'] / full.groupby('product_category')['product_price'].transform('median')

full['store_age_bucket'] = pd.cut(full['store_age_years'], bins=[0, 20, 30, 40, 100],
                                   labels=['New', 'Mid', 'Mature', 'Old'])
full['was_zero_visibility'] = (full['shelf_visibility_raw'] == 0).astype(int)
full['price_bucket'] = pd.qcut(full['product_price'], 10, labels=False, duplicates='drop')

# safe popularity/assortment signals (no target used)
full['product_store_count'] = full.groupby('product_code')['product_code'].transform('count')
full['store_assortment_size'] = full.groupby('store_code')['product_category'].transform('nunique')
full['category_size'] = full.groupby('product_category')['product_category'].transform('count')

size_rank = {'Unknown': -1, 'Small': 0, 'Medium': 1, 'Large': 2}
full['store_size_rank'] = full['store_size'].map(size_rank)
tier_rank = {'Tier_1': 1, 'Tier_2': 2, 'Tier_3': 3}
full['store_location_tier_rank'] = full['store_location_tier'].map(tier_rank)
full['size_x_tier'] = full['store_size_rank'] * full['store_location_tier_rank']

cat_cols = ['product_code', 'fat_content', 'product_category', 'store_code',
            'store_size', 'store_location_tier', 'store_format', 'item_type_broad',
            'store_age_bucket']
for c in cat_cols:
    full[c] = full[c].astype(str)

full['store_x_category'] = full['store_code'] + '_' + full['product_category']
full['format_x_category'] = full['store_format'] + '_' + full['product_category']
full['tier_x_category'] = full['store_location_tier'] + '_' + full['product_category']

In [ ]:
train_fe = full[full['is_train'] == 1].reset_index(drop=True)
test_fe = full[full['is_train'] == 0].reset_index(drop=True)

y = np.log1p(train_fe['total_sales'].values)

TE_COLS = ['product_code', 'product_category', 'store_code', 'store_format',
           'store_location_tier', 'store_size', 'item_type_broad',
           'store_x_category', 'format_x_category', 'tier_x_category']

NUM_COLS = ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years',
            'price_per_kg', 'log_price', 'log_weight', 'visibility_mean_ratio',
            'visibility_rank_in_store', 'price_rank_in_category', 'price_vs_category_median',
            'was_zero_visibility', 'store_size_rank', 'store_location_tier_rank',
            'size_x_tier', 'price_bucket', 'product_store_count',
            'store_assortment_size', 'category_size']

GLOBAL_MEAN = y.mean()

def kfold_target_encode(train_df, test_df, col, y, n_folds=N_FOLDS, smoothing=15):
    oof = np.zeros(len(train_df))
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RNG)
    for tr_idx, val_idx in kf.split(train_df):
        stats = pd.DataFrame({col: train_df[col].values[tr_idx], 'y': y[tr_idx]}).groupby(col)['y'].agg(['mean', 'count'])
        smooth = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        oof[val_idx] = pd.Series(train_df[col].values[val_idx]).map(smooth).fillna(GLOBAL_MEAN).values
    stats_full = pd.DataFrame({col: train_df[col].values, 'y': y}).groupby(col)['y'].agg(['mean', 'count'])
    smooth_full = (stats_full['mean'] * stats_full['count'] + GLOBAL_MEAN * smoothing) / (stats_full['count'] + smoothing)
    test_enc = test_df[col].map(smooth_full).fillna(GLOBAL_MEAN).values
    return oof, test_enc

for col in TE_COLS:
    oof_a, test_a = kfold_target_encode(train_fe, test_fe, col, y, smoothing=10)
    oof_b, test_b = kfold_target_encode(train_fe, test_fe, col, y, smoothing=50)
    train_fe[f'te_{col}'] = (oof_a + oof_b) / 2
    test_fe[f'te_{col}'] = (test_a + test_b) / 2

TE_FEATURES = [f'te_{c}' for c in TE_COLS]
FEATURES = NUM_COLS + TE_FEATURES

X = train_fe[FEATURES].values
X_test = test_fe[FEATURES].values
print("Feature matrix:", X.shape, "| Test matrix:", X_test.shape)

Feature matrix: (6818, 29) | Test matrix: (1705, 29)


In [ ]:
def get_models(seed):
    models = {}
    models['hgb'] = HistGradientBoostingRegressor(
        max_iter=700, learning_rate=0.025, max_depth=6, max_leaf_nodes=31,
        l2_regularization=1.0, min_samples_leaf=20,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=seed
    )
    models['et'] = ExtraTreesRegressor(
        n_estimators=700, max_depth=14, min_samples_leaf=3, max_features=0.7,
        n_jobs=-1, random_state=seed
    )
    models['ridge'] = Ridge(alpha=5.0, random_state=seed)
    if HAS_LGB:
        models['lgb'] = lgb.LGBMRegressor(
            n_estimators=4000, learning_rate=0.01, num_leaves=31, max_depth=-1,
            min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.6, random_state=seed, n_jobs=-1, verbose=-1
        )
    if HAS_XGB:
        models['xgb'] = xgb.XGBRegressor(
            n_estimators=4000, learning_rate=0.01, max_depth=6, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, tree_method='hist',
            eval_metric='rmse', early_stopping_rounds=150
        )
    if HAS_CAT:
        models['cat'] = CatBoostRegressor(
            iterations=4000, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            random_seed=seed, loss_function='RMSE', verbose=False
        )
    return models

def fit_predict(name, model, X_tr, y_tr, X_val, y_val, X_test):
    if name == 'lgb':
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
    elif name == 'xgb':
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    elif name == 'cat':
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150, verbose=False)
    else:
        model.fit(X_tr, y_tr)
    return model.predict(X_val), model.predict(X_test)

model_names = list(get_models(42).keys())
oof_store = {m: np.zeros(len(train_fe)) for m in model_names}
test_store = {m: np.zeros(len(test_fe)) for m in model_names}

for seed in SEEDS:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        models = get_models(seed)
        for name, model in models.items():
            p_val, p_test = fit_predict(name, model, X_tr, y_tr, X_val, y_val, X_test)
            oof_store[name][val_idx] += p_val / len(SEEDS)
            test_store[name] += p_test / (len(SEEDS) * N_FOLDS)
    print(f"seed {seed} done")

for name in model_names:
    rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_store[name])))
    print(f"{name:6s} OOF RMSE: {rmse:.3f}")

seed 42 done
seed 202 done
seed 777 done
hgb    OOF RMSE: 1116.860
et     OOF RMSE: 1121.112
ridge  OOF RMSE: 1102.825
lgb    OOF RMSE: 1118.656
xgb    OOF RMSE: 1135.269


In [ ]:
oof_matrix = np.column_stack([oof_store[m] for m in model_names])
test_matrix = np.column_stack([test_store[m] for m in model_names])
y_true = np.expm1(y)

def blend_rmse(w):
    w = np.abs(w) / np.abs(w).sum()
    pred = np.expm1(oof_matrix @ w)
    return np.sqrt(mean_squared_error(y_true, pred))

res = minimize(blend_rmse, x0=np.ones(len(model_names)) / len(model_names), method='Nelder-Mead')
best_w = np.abs(res.x) / np.abs(res.x).sum()

print("Blend weights:")
for m, w in zip(model_names, best_w):
    print(f"  {m:6s}: {w:.3f}")

final_oof_rmse = blend_rmse(res.x)
print(f"\nFINAL BLENDED OOF RMSE: {final_oof_rmse:.3f}")

baseline_rmse = np.sqrt(mean_squared_error(train_fe['total_sales'], np.full(len(train_fe), train_fe['total_sales'].mean())))
print(f"Naive baseline RMSE: {baseline_rmse:.3f}  |  Improvement: {(1-final_oof_rmse/baseline_rmse)*100:.1f}%")

test_log_pred = test_matrix @ best_w
final_preds = np.clip(np.expm1(test_log_pred), 0, None)

Blend weights:
  hgb   : 0.000
  et    : 0.089
  ridge : 0.908
  lgb   : 0.000
  xgb   : 0.003

FINAL BLENDED OOF RMSE: 1102.715
Naive baseline RMSE: 1697.770  |  Improvement: 35.0%
